# Redbus Journey Analysis

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow
!pip install xgboost lightgbm catboost optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 16.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
#Load path where my dataset is
path = "/content/drive/MyDrive/RedBusAnalysis"

In [4]:
#Importing pandas and numpy
import numpy as np
import pandas as pd

In [5]:
#Loading data set
train_data = pd.read_csv(path + "/train.csv")
transaction_data = pd.read_csv(path + "/transactions.csv")
test_data = pd.read_csv(path + "/test_8gqdJqH.csv")

# Feature Extraction

In [6]:
#Filtering for prediction 15 days before journey
transaction_15 = transaction_data[transaction_data["dbd"] == 15]

In [7]:
#Creating unique route key to match later with test dataset
transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)

/tmp/ipython-input-7-1703587420.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)


In [8]:
transaction_15 = transaction_15.dropna()

In [9]:
#Selecting Relevant feature
features = transaction_15[["route_key", "cumsum_seatcount", "cumsum_searchcount", "srcid_region", "destid_region", "srcid_tier", "destid_tier"]]

In [10]:
#Merge with train labels
train_data["route_key"] = train_data["doj"].astype(str) + "_" + train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
#Drop if existing feature in train data to avoid collision during merge
cols_to_drop = ["cumsum_seatcount", "cumsum_searchcount",
                "srcid_region", "destid_region", "srcid_tier", "destid_tier"]

train_data = train_data.drop(columns=[col for col in cols_to_drop if col in train_data.columns])

train_data = train_data.merge(features, on="route_key", how="left")
train_data.dropna(inplace=True)

In [11]:
#Mergin with test set
duplicate_cols = [
    "cumsum_seatcount", "cumsum_searchcount",
    "srcid_region", "destid_region",
    "srcid_tier", "destid_tier"
]

# Drop them from test_data if they exist
test_data = test_data.drop(columns=[col for col in duplicate_cols if col in test_data.columns])
test_data = test_data.merge(features, on="route_key", how="left")
test_data['cumsum_seatcount'] = test_data['cumsum_seatcount'].fillna(0)
test_data['cumsum_searchcount'] = test_data['cumsum_searchcount'].fillna(0)
for col in ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]:
    test_data[col] = test_data[col].fillna("Unknown")

In [12]:
#Adding seat search ratio -- features
train_data["seat_to_search_ratio"] = train_data["cumsum_seatcount"] / (train_data["cumsum_searchcount"] + 1)
test_data["seat_to_search_ratio"] = test_data["cumsum_seatcount"] / (test_data["cumsum_searchcount"] + 1)

In [13]:
#Log Transform (Stabilize scale) -- features
for df in [train_data, test_data]:
    df["log_seatcount"] = np.log1p(df["cumsum_seatcount"])  # log(1 + x)
    df["log_searchcount"] = np.log1p(df["cumsum_searchcount"])

In [14]:
#Extract day of week and Month from object
train_data["doj"] = pd.to_datetime(train_data["doj"])
test_data["doj"] = pd.to_datetime(test_data["doj"])

# Extract calendar features
for df in [train_data, test_data]:
    df["doj_dayofweek"] = df["doj"].dt.dayofweek  # 0=Monday
    df["doj_month"] = df["doj"].dt.month

In [15]:
#String based interaction (categorical model can handle it)
# Label encode it (or use frequency encoding)
train_data["route_pair"] = train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
test_data["route_pair"] = test_data["srcid"].astype(str) + "_" + test_data["destid"].astype(str)

# Label encode it (or use frequency encoding)
from sklearn.preprocessing import LabelEncoder
le_route = LabelEncoder()
combined_routes = pd.concat([train_data["route_pair"], test_data["route_pair"]])
le_route.fit(combined_routes)

train_data["route_pair_enc"] = le_route.transform(train_data["route_pair"])
test_data["route_pair_enc"] = le_route.transform(test_data["route_pair"])

In [16]:
#Option B: Hash trick (useful if lots of unique routes)
train_data["route_hash"] = train_data["route_pair"].apply(lambda x: hash(x) % 1000)
test_data["route_hash"] = test_data["route_pair"].apply(lambda x: hash(x) % 1000)

In [17]:
#Target Encoding (Mean Encoding)
def target_encode(train_df, test_df, cat_col, target_col="final_seatcount"):
    target_map = train_df.groupby(cat_col)[target_col].mean()
    train_df[f"{cat_col}_target"] = train_df[cat_col].map(target_map)
    test_df[f"{cat_col}_target"] = test_df[cat_col].map(target_map)  # Use same map (no leakage)
    test_df[f"{cat_col}_target"] = test_df[f"{cat_col}_target"].fillna(target_map.mean())

In [18]:
categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier", "route_pair"]
for col in categorical_features:
    target_encode(train_data, test_data, col)

In [19]:
#Frequency Encoding
def frequency_encode(train_df, test_df, col):
    freq_map = train_df[col].value_counts(normalize=False)
    train_df[f"{col}_freq"] = train_df[col].map(freq_map)
    test_df[f"{col}_freq"] = test_df[col].map(freq_map)
    test_df[f"{col}_freq"] = test_df[f"{col}_freq"].fillna(0)  # Handle unseen

In [20]:
for col in categorical_features:
    frequency_encode(train_data, test_data, col)

In [21]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [22]:
# Encoding categorical features safely
# categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]
# for col in categorical_features:
#     le = LabelEncoder()

#     # Combine train + test categories for fitting
#     combined_values = pd.concat([train_data[col], test_data[col]], axis=0).astype(str)

#     # Fit encoder on all possible values
#     le.fit(combined_values)

#     # Transform separately
#     train_data[col] = le.transform(train_data[col].astype(str))
#     test_data[col] = le.transform(test_data[col].astype(str))

In [23]:
#Select final features
features = [
    "cumsum_seatcount", "cumsum_searchcount", "seat_to_search_ratio",
    "log_seatcount", "log_searchcount",
    "doj_dayofweek", "doj_month",
    "srcid_region_target", "destid_region_target",
    "srcid_tier_target", "destid_tier_target",
    "route_pair_target"
]
# features = [
#     "cumsum_seatcount", "cumsum_searchcount",
#     "seat_to_search_ratio", "log_seatcount", "log_searchcount",
#     "doj_dayofweek", "doj_month",
#     "srcid_region", "destid_region", "srcid_tier", "destid_tier",
#     "route_pair_enc"
# ]
target = "final_seatcount"

In [24]:
X = train_data[features]
y = train_data[target]
X_test = test_data[features]

In [25]:
#Normalizing features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Model Training

In [26]:
#Split data for Optuna
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [27]:
#Optuna Hyperparameter Tuning for XGBoost
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
    }

    model = XGBRegressor(**params)

    # Use early stopping with evaluation set
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        #early_stopping_rounds=50,
        verbose=False
    )
    #model = XGBRegressor(**params, random_state=42)
    #model.fit(X_train, y_train)

    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

best_params = study.best_params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)
xgb_pred = best_model.predict(X_test)


[I 2025-06-22 05:35:58,723] A new study created in memory with name: no-name-7cfcac24-5b33-481b-855f-a1f0d1aeabe7
[I 2025-06-22 05:36:05,014] Trial 0 finished with value: 446.8900646861637 and parameters: {'n_estimators': 536, 'max_depth': 8, 'learning_rate': 0.15524594384392063, 'subsample': 0.6768758594247978, 'colsample_bytree': 0.8038710548654318, 'gamma': 0.7111541940979821, 'reg_alpha': 0.49902309144078505, 'reg_lambda': 4.259397711482182}. Best is trial 0 with value: 446.8900646861637.
[I 2025-06-22 05:36:23,734] Trial 1 finished with value: 434.06688392011125 and parameters: {'n_estimators': 626, 'max_depth': 10, 'learning_rate': 0.010579944657934895, 'subsample': 0.919771060164248, 'colsample_bytree': 0.8392843372571843, 'gamma': 2.3415922764111805, 'reg_alpha': 1.9478791134570113, 'reg_lambda': 1.613113200952725}. Best is trial 1 with value: 434.06688392011125.
[I 2025-06-22 05:37:13,374] Trial 2 finished with value: 442.6902657997463 and parameters: {'n_estimators': 1435, 'm

# Sequential input preparation

In [28]:
#Select dbd from 1 to 30
ts_data = transaction_data[transaction_data["dbd"].between(1,30)].copy()

In [29]:
#Create route key
ts_data["route_key"] = ts_data["doj"] + "_" + ts_data["srcid"].astype(str) + "_" + ts_data["destid"].astype(str)

In [30]:
#Pivot to time series format
#pivot 2 features per day
pivot_seat = ts_data.pivot(index="route_key", columns="dbd", values="cumsum_seatcount")
pivot_search = ts_data.pivot(index="route_key", columns="dbd", values="cumsum_searchcount")

#Rename columns for clarity
pivot_seat.columns = [f"seat_dbd{col}" for col in pivot_seat.columns]
pivot_search.columns = [f"search_dbd{col}" for col in pivot_search.columns]

#Join features horizontally
ts_features = pd.concat([pivot_seat, pivot_search], axis = 1).fillna(0)

In [31]:
#Merge with Traib Labels
lstm_train = train_data[["route_key", "final_seatcount"]].merge(ts_features, on="route_key", how="inner")

In [32]:
#Reshape to 3D for LSTM
X_lstm = lstm_train.drop(columns=["route_key", "final_seatcount"]).values
y_lstm = lstm_train["final_seatcount"].values

# Reshape: (samples, time_steps=30, features=2)
X_lstm = X_lstm.reshape((X_lstm.shape[0], 30, 2))

# LSTM Training

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

X_train_lstm, X_val_lstm, y_train_lstm, y_val_lstm = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42)

model = Sequential([
    LSTM(64, input_shape=(30, 2), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(loss='mse', optimizer='adam')
model.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=32, validation_data=(X_val_lstm, y_val_lstm))

Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1680/1680 ━━━━━━━━━━━━━━━━━━━━ 28s 15ms/step - loss: 3744319.5000 - val_loss: 1040291.4375
Epoch 2/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - loss: 1022600.0000 - val_loss: 781119.6250
Epoch 3/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 41s 16ms/step - loss: 784437.3125 - val_loss: 615363.8125
Epoch 4/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 41s 16ms/step - loss: 726353.8125 - val_loss: 498256.0000
Epoch 5/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 39s 15ms/step - loss: 649566.6250 - val_loss: 487619.0000
Epoch 6/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 628696.4375 - val_loss: 504470.9062
Epoch 7/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 38s 15ms/step - loss: 615520.5000 - val_loss: 419446.9688
Epoch 8/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 651327.0000 - val_loss: 549418.1875
Epoch 9/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 696323.0000 - val_loss: 599672.0625
Epoch 10/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 662752.1250 - val_loss: 458081.0312
Epoch 1

In [49]:
#Reshaping test data
# Filter transactions for test routes only
test_routes = test_data["route_key"].unique()
ts_test = ts_data[ts_data["route_key"].isin(test_routes)]


# Pivot test features
pivot_seat_test = ts_test.pivot(index="route_key", columns="dbd", values="cumsum_seatcount")
pivot_search_test = ts_test.pivot(index="route_key", columns="dbd", values="cumsum_searchcount")

pivot_seat_test.columns = [f"seat_dbd{c}" for c in pivot_seat_test.columns]
pivot_search_test.columns = [f"search_dbd{c}" for c in pivot_search_test.columns]

ts_test_features = pd.concat([pivot_seat_test, pivot_search_test], axis=1)
ts_test_features = ts_test_features.fillna(0)


# Merge with test_data to preserve original ordering
lstm_test = test_data[["route_key"]].merge(ts_test_features, on="route_key", how="left")

X_test_lstm = lstm_test.drop(columns=["route_key"]).values
X_test_lstm = X_test_lstm.reshape((X_test_lstm.shape[0], 30, 2))
X_test_lstm = np.nan_to_num(X_test_lstm, nan=0.0)

In [50]:
lstm_pred = model.predict(X_test_lstm).flatten()

185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [84]:
lstm_pred.shape

(5900,)

In [52]:
#Model 2: LGBMRegressor
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(n_estimators=100, random_state=42)
lgb.fit(X_train, y_train)
lgb_pred = lgb.predict(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001972 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 53760, number of used features: 12
[LightGBM] [Info] Start training from score 2003.632533


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# Building Ensemble Model

In [145]:
#Ensembling using cut_sum weight
weights = np.array([0.5, 0.3, 0.2])  # Your choice
final_pred = (
    weights[0] * xgb_pred +
    weights[1] * lgb_pred +
    weights[2] * lstm_pred
)


In [146]:
submission = test_data[["route_key"]].copy()
submission["final_seatcount"] = final_pred.round().astype(int)
submission.to_csv("submission_file.csv", index=False)

In [147]:
#Download submission file
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [148]:
submission

,route_key,final_seatcount
0,2025-02-11_46_45,3066
1,2025-01-20_17_23,1403
2,2025-01-08_02_14,890
3,2025-01-08_08_47,870
4,2025-01-08_09_46,2375
...,...,...
5895,2025-01-23_46_48,3840
5896,2025-02-21_46_09,2438
5897,2025-01-17_32_19,1886
5898,2025-01-24_45_03,941
